# 03 – Time Series Demand Forecasting

This notebook will explore time series models and workflows for demand forecasting using the `src/forecasting` and `src/data` packages.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.synthetic_generators import generate_demand_series
from src.forecasting.time_series_models import train_arima, forecast_horizon
from src.optimization.resource_allocation import optimize_horizon_from_forecast

%matplotlib inline

In [ ]:
# Generate synthetic daily demand and split into history and forecast horizon
n_periods = 200
horizon = 30

values = generate_demand_series(n_periods, random_state=0)
index = pd.date_range(start="2024-01-01", periods=n_periods, freq="D")
demand_series = pd.Series(values, index=index, name="demand")

history = demand_series.iloc[:-horizon]
actual_future = demand_series.iloc[-horizon:]

In [ ]:
# Plot historical demand
ax = history.plot(figsize=(10, 4), title="Synthetic historical demand")
ax.set_ylabel("Units")
plt.show()

In [ ]:
# Train ARIMA on the historical demand and forecast the next horizon
model = train_arima(history, order=(2, 1, 2))
forecast = forecast_horizon(model, steps=horizon)

# Plot history, actual future, and forecast
ax = history.plot(label="history", figsize=(10, 4))
actual_future.plot(ax=ax, label="actual future")
forecast.plot(ax=ax, label="forecast", linestyle="--")
ax.set_title("Demand forecast with ARIMA (history vs actual vs forecast)")
ax.set_ylabel("Units")
ax.legend()
plt.show()

In [ ]:
# Optimize resource allocation for each period in the forecast horizon
capacities = np.array([150.0, 120.0, 80.0])
unit_cost = 1.0

plan = optimize_horizon_from_forecast(
    forecast=forecast,
    costs=unit_cost,
    capacities=capacities,
)

plan.allocations.shape, plan.objective_values.shape

In [ ]:
# Inspect a few periods of the horizon plan
print("First 5 objective values:")
print(plan.objective_values[:5])
print("First 5 allocation rows (period x resources):")
print(plan.allocations[:5])

In [ ]:
# Plot allocation per resource per period over the forecast horizon
alloc_df = pd.DataFrame(
    plan.allocations,
    index=forecast.index,
    columns=[f"R{i+1}" for i in range(len(capacities))],
)

ax = alloc_df.plot(figsize=(10, 4))
ax.set_title("Allocation per resource over forecast horizon")
ax.set_ylabel("Units")
plt.show()

In [ ]:
# Plot cost per period over the forecast horizon
cost_series = pd.Series(plan.objective_values, index=forecast.index, name="cost")
ax = cost_series.plot(figsize=(10, 4))
ax.set_title("Cost per period over forecast horizon")
ax.set_ylabel("Cost")
plt.show()

In [ ]:
# Plot the forecast curve alone for clarity
ax = forecast.plot(figsize=(10, 4))
ax.set_title("Forecast demand over horizon")
ax.set_ylabel("Units")
plt.show()